In [ ]:
"""
Generate synthetic QA pairs from chunks, with
RAFT-style distractors — using a model running locally on a Kaggle GPU
instead of a rate-limited API.

Reads chunks.jsonl (from chunking step), runs a local 4-bit Qwen2.5-7B-Instruct
model to generate a question + grounded answer per chunk, samples hard +
easy distractors, and writes a raft_dataset.jsonl ready for SFT-style
fine-tuning (instruction/context/question/answer fields).
"""

#!pip install -q transformers accelerate bitsandbytes

import json
import os
import random
import re
import time
from collections import defaultdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# CONFIG
CHUNKS_PATH = "/kaggle/input/datasets/chunks.jsonl"  
OUT_PATH = "/kaggle/working/raft_dataset.jsonl"               
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
NUM_DISTRACTORS = 3                  
HARD_DISTRACTOR_RATIO = 0.5         
ABSTENTION_FRACTION = 0.12           
MULTIHOP_FRACTION = 0.2              
MAX_RETRIES = 3                     
MAX_NEW_TOKENS = 400                 
TEMPERATURE = 0.6
CHECKPOINT_EVERY = 20

In [ ]:
print("Loading model in 4-bit (this takes a minute or two)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",  
)
model.eval()
print("Model loaded.")

In [ ]:
# LLM call wrapper (local generation, not an API call)
def call_llm(system: str, user: str) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    for attempt in range(MAX_RETRIES):
        try:
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            # Slice off the input prompt tokens — we only want the new completion.
            new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
            decoded = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
            return decoded
        except torch.cuda.OutOfMemoryError as e:
            print(f"  !! CUDA OOM on attempt {attempt+1}/{MAX_RETRIES}: {e}")
            torch.cuda.empty_cache()
            time.sleep(2)
        except Exception as e:
            print(f"  !! generation error (attempt {attempt+1}/{MAX_RETRIES}): {e}")
            time.sleep(1)

    raise RuntimeError("Local generation failed after retries")


def extract_json(text: str) -> dict:
    """
    Parse the model's JSON response. Local models (no response_schema
    support) are more likely than Gemini's structured-output mode to wrap
    output in markdown fences, add a stray preamble sentence, or trail off
    with extra text after the JSON object — so this is more defensive than
    the API version: strips fences, then falls back to extracting the
    first {...} block if a direct parse fails.
    """
    cleaned = re.sub(r"^```json\s*|^```\s*|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    # Fallback: find the first balanced-looking {...} block in the text,
    # in case the model added a preamble/trailing sentence around the JSON.
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        return json.loads(match.group(0))
    raise json.JSONDecodeError("No JSON object found", cleaned, 0)

In [ ]:
# QA generation prompts
SINGLE_CHUNK_SYSTEM = """You are generating training data for a research assistant chatbot \
specializing in microfluidics and biosensing. You produce one high-quality \
question-answer pair fully grounded in a given excerpt. Always set "skip" to \
false for this task — it only applies to a different task type. Respond with \
ONLY a single valid JSON object and nothing else — no preamble, no markdown \
fences, no explanation before or after."""

SINGLE_CHUNK_TEMPLATE = """Given the following excerpt from a research paper, generate ONE question-answer pair.

PAPER: {paper_title}
SECTION: {section}
EXCERPT:
{chunk_text}

Requirements:
- The question should be the kind a graduate researcher would genuinely ask while working on a related project. Vary the type across calls: factual recall, mechanistic ("how/why does X happen"), or comparative.
- The answer must be fully grounded in the excerpt. Do not introduce outside facts.
- Include a short supporting paraphrase from the excerpt as evidence (do not quote verbatim more than a few words).
- Keep the answer concise (2-4 sentences).

Respond with ONLY this JSON object, filled in, and nothing else:
{{"skip": false, "question": "...", "answer": "...", "evidence_span": "..."}}"""

MULTIHOP_SYSTEM = """You are generating training data for a research assistant chatbot \
specializing in microfluidics and biosensing. You produce questions that require \
synthesizing information across two related excerpts. If the two excerpts given to \
you are not meaningfully related, set "skip" to true and leave the other fields null \
instead of forcing a question. Respond with ONLY a single valid JSON object and \
nothing else — no preamble, no markdown fences, no explanation before or after."""

MULTIHOP_TEMPLATE = """Given these two excerpts from related research papers, generate ONE \
question that can only be answered by combining information from BOTH excerpts \
(comparison, contrast, or synthesis). If the two excerpts are not meaningfully related, \
set "skip" to true and leave "question"/"answer"/"evidence_span" as null.

EXCERPT A (from "{paper_a}", section {section_a}):
{chunk_a}

EXCERPT B (from "{paper_b}", section {section_b}):
{chunk_b}

Respond with ONLY a JSON object in one of these two exact shapes, and nothing else:
{{"skip": false, "question": "...", "answer": "...", "evidence_span": "..."}}
{{"skip": true, "question": null, "answer": null, "evidence_span": null}}"""


def generate_single_chunk_qa(chunk: dict) -> dict | None:
    user_prompt = SINGLE_CHUNK_TEMPLATE.format(
        paper_title=chunk["paper_title"],
        section=chunk["section"],
        chunk_text=chunk["text"],
    )
    for attempt in range(MAX_RETRIES):
        raw = call_llm(SINGLE_CHUNK_SYSTEM, user_prompt)
        try:
            parsed = extract_json(raw)
        except json.JSONDecodeError:
            print(f"  !! JSON parse failed for chunk {chunk['chunk_id']} (attempt {attempt+1}/{MAX_RETRIES})")
            continue
        if parsed.get("skip") or not parsed.get("question"):
            print(f"  !! model unexpectedly skipped single-chunk QA for {chunk['chunk_id']}")
            return None
        return parsed
    print(f"  !! giving up on chunk {chunk['chunk_id']} after {MAX_RETRIES} failed parse attempts")
    return None


def generate_multihop_qa(chunk_a: dict, chunk_b: dict) -> dict | None:
    user_prompt = MULTIHOP_TEMPLATE.format(
        paper_a=chunk_a["paper_title"], section_a=chunk_a["section"], chunk_a=chunk_a["text"],
        paper_b=chunk_b["paper_title"], section_b=chunk_b["section"], chunk_b=chunk_b["text"],
    )
    for attempt in range(MAX_RETRIES):
        raw = call_llm(MULTIHOP_SYSTEM, user_prompt)
        try:
            parsed = extract_json(raw)
        except json.JSONDecodeError:
            print(f"  !! multi-hop JSON parse failed (attempt {attempt+1}/{MAX_RETRIES})")
            continue
        if parsed.get("skip") or not parsed.get("question"):
            return None
        return parsed
    return None

In [ ]:
# Distractor sampling (the RAFT-specific part — unchanged from the API version)

def sample_distractors(golden_chunks: list[dict], all_chunks: list[dict], n: int) -> list[dict]:
    """
    Sample n distractor chunks for the given golden chunk(s).
    Half "hard" (same paper, different chunk) + half "easy" (different paper, random).
    """
    golden_ids = {c["chunk_id"] for c in golden_chunks}
    golden_papers = {c["paper_title"] for c in golden_chunks}

    same_paper_pool = [c for c in all_chunks if c["paper_title"] in golden_papers and c["chunk_id"] not in golden_ids]
    other_paper_pool = [c for c in all_chunks if c["paper_title"] not in golden_papers]

    n_hard = max(1, round(n * HARD_DISTRACTOR_RATIO)) if same_paper_pool else 0
    n_hard = min(n_hard, len(same_paper_pool))
    n_easy = n - n_hard

    distractors = []
    if n_hard:
        distractors.extend(random.sample(same_paper_pool, n_hard))
    if n_easy and other_paper_pool:
        n_easy = min(n_easy, len(other_paper_pool))
        distractors.extend(random.sample(other_paper_pool, n_easy))

    # Backfill if we came up short (small corpus edge case)
    while len(distractors) < n and len(distractors) < len(all_chunks) - len(golden_ids):
        candidate = random.choice(all_chunks)
        if candidate["chunk_id"] not in golden_ids and candidate not in distractors:
            distractors.append(candidate)

    return distractors[:n]


RAFT_INSTRUCTION = (
    "Answer the question using only the provided documents. "
    "Cite which document(s) support your answer. "
    "If the documents don't contain the answer, say so explicitly rather than guessing."
)

ABSTENTION_ANSWER_TEMPLATE = (
    "The provided documents do not contain information to answer this question. "
    "I'd need a source discussing {topic_hint} to answer this accurately."
)


def assemble_example(question: str, answer: str, golden_chunks: list[dict],
                      distractors: list[dict], is_abstention: bool = False) -> dict:
    """Build one RAFT-format training example with shuffled document order."""
    doc_pool = golden_chunks + distractors
    random.shuffle(doc_pool)

    context_parts = []
    golden_doc_labels = []
    for i, c in enumerate(doc_pool, start=1):
        label = f"DOC {i}"
        context_parts.append(f"[{label}] ({c['paper_title']}, {c['section']}) {c['text']}")
        if c in golden_chunks:
            golden_doc_labels.append(label)

    return {
        "instruction": RAFT_INSTRUCTION,
        "context": "\n\n".join(context_parts),
        "question": question,
        "answer": answer,
        "metadata": {
            "golden_doc_labels": golden_doc_labels,
            "source_papers": list({c["paper_title"] for c in golden_chunks}),
            "is_abstention": is_abstention,
            "num_docs": len(doc_pool),
        },
    }

In [ ]:
# Checkpointing — write progress incrementally, not just at the end

def append_to_checkpoint(examples: list[dict], path: str) -> None:
    """Append newly generated examples to disk immediately, so a Kaggle
    session timeout or crash only loses the last few examples in progress,
    not the entire run."""
    with open(path, "a") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")


def load_existing_chunk_ids(path: str) -> set[str]:
    """
    On resume, figure out which chunks already have a generated example so
    we don't regenerate (and waste GPU time on) the same ones. Matches on
    the golden chunk_id(s) embedded in each example's metadata-adjacent
    context labels isn't reliable, so instead we tag every example with the
    source chunk_id(s) explicitly for this purpose.
    """
    if not os.path.exists(path):
        return set()
    seen = set()
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            ex = json.loads(line)
            for cid in ex.get("metadata", {}).get("source_chunk_ids", []):
                seen.add(cid)
    return seen

In [ ]:
# Main generation loop

def load_chunks(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def group_by_paper(chunks: list[dict]) -> dict[str, list[dict]]:
    groups = defaultdict(list)
    for c in chunks:
        groups[c["paper_title"]].append(c)
    return groups


def main():
    all_chunks = load_chunks(CHUNKS_PATH)
    print(f"Loaded {len(all_chunks)} chunks from {CHUNKS_PATH}")
    by_paper = group_by_paper(all_chunks)

    # RESUME: if OUT_PATH already has examples from a previous (interrupted)
    # run, skip chunks we've already generated QA for, rather than starting
    # over and burning GPU time re-doing work that's already on disk.
    already_done_chunk_ids = load_existing_chunk_ids(OUT_PATH)
    if already_done_chunk_ids:
        print(f"Resuming: found {len(already_done_chunk_ids)} chunks already processed in {OUT_PATH}")

    pending_checkpoint = []  # examples not yet flushed to disk

    def maybe_checkpoint(force: bool = False):
        nonlocal pending_checkpoint
        if pending_checkpoint and (force or len(pending_checkpoint) >= CHECKPOINT_EVERY):
            append_to_checkpoint(pending_checkpoint, OUT_PATH)
            print(f"  [checkpoint] wrote {len(pending_checkpoint)} examples to disk")
            pending_checkpoint = []

    # --- Pass 1: single-chunk factual/mechanistic QA -----------------------
    print("\n--- Pass 1: single-chunk QA generation ---")
    n_multihop = round(len(all_chunks) * MULTIHOP_FRACTION)
    n_single = len(all_chunks) - n_multihop

    single_chunks = [c for c in all_chunks if c["chunk_id"] not in already_done_chunk_ids]
    single_chunks = random.sample(single_chunks, min(n_single, len(single_chunks)))

    single_count = 0
    for i, chunk in enumerate(single_chunks):
        print(f"[{i+1}/{len(single_chunks)}] {chunk['paper_title']} / {chunk['section']}")
        qa = generate_single_chunk_qa(chunk)
        if qa is None:
            continue
        distractors = sample_distractors([chunk], all_chunks, NUM_DISTRACTORS)
        example = assemble_example(qa["question"], qa["answer"], [chunk], distractors)
        example["metadata"]["source_chunk_ids"] = [chunk["chunk_id"]]
        pending_checkpoint.append(example)
        single_count += 1
        maybe_checkpoint()
    maybe_checkpoint(force=True)

    # --- Pass 2: multi-hop QA across related chunks -------------------------
    print("\n--- Pass 2: multi-hop QA generation ---")
    papers = list(by_paper.keys())
    attempts = 0
    generated = 0
    while generated < n_multihop and attempts < n_multihop * 3:
        attempts += 1
        if len(papers) >= 2:
            p_a, p_b = random.sample(papers, 2)
        else:
            p_a = p_b = papers[0]
        chunk_a = random.choice(by_paper[p_a])
        chunk_b = random.choice(by_paper[p_b])
        if chunk_a["chunk_id"] == chunk_b["chunk_id"]:
            continue
        print(f"[{generated+1}/{n_multihop}] {p_a} + {p_b}")
        qa = generate_multihop_qa(chunk_a, chunk_b)
        if qa is None:
            continue
        golden = [chunk_a, chunk_b]
        distractors = sample_distractors(golden, all_chunks, NUM_DISTRACTORS)
        example = assemble_example(qa["question"], qa["answer"], golden, distractors)
        example["metadata"]["source_chunk_ids"] = [chunk_a["chunk_id"], chunk_b["chunk_id"]]
        pending_checkpoint.append(example)
        generated += 1
        maybe_checkpoint()
    maybe_checkpoint(force=True)

    # --- Pass 3: abstention examples (all distractors, no golden chunk) ----
    print("\n--- Pass 3: abstention examples ---")
    # Re-read everything written so far (this run + any resumed prior run)
    # so abstention sampling draws from the full dataset, not just this
    # session's in-memory additions.
    full_dataset_so_far = load_chunks(OUT_PATH) if os.path.exists(OUT_PATH) else []
    n_abstain = round(len(full_dataset_so_far) * ABSTENTION_FRACTION)
    non_abstain = [ex for ex in full_dataset_so_far if not ex["metadata"]["is_abstention"]]
    abstain_candidates = random.sample(non_abstain, min(n_abstain, len(non_abstain)))

    for ex in abstain_candidates:
        topic_hint = ex["metadata"]["source_papers"][0] if ex["metadata"]["source_papers"] else "this topic"
        fully_distractor_docs = sample_distractors([], all_chunks, NUM_DISTRACTORS + 1)
        new_example = assemble_example(
            question=ex["question"],
            answer=ABSTENTION_ANSWER_TEMPLATE.format(topic_hint=topic_hint),
            golden_chunks=[],
            distractors=fully_distractor_docs,
            is_abstention=True,
        )
        new_example["metadata"]["source_chunk_ids"] = []  # abstention examples have no golden source
        pending_checkpoint.append(new_example)
        maybe_checkpoint()
    maybe_checkpoint(force=True)

    final_dataset = load_chunks(OUT_PATH)
    print(f"\nDone. {len(final_dataset)} total RAFT examples in {OUT_PATH}")
    print(f"  - single-chunk generated this run: {single_count}")
    print(f"  - multi-hop generated this run: {generated}")
    print(f"  - abstention generated this run: {len(abstain_candidates)}")
    print("\nSample example:")
    print(json.dumps(final_dataset[0], indent=2)[:800])


if __name__ == "__main__":
    main()